# D\&R Replication Guide

**Debate & Reflect — Knowledge Distillation on MMLU-Pro Computer Science**

This notebook walks through the full replication: environment setup → multi-agent debate → SFT + T-DPO training → evaluation.

**Run this notebook inside WSL (Ubuntu).** Open a terminal, activate your conda environment, then launch Jupyter:

```bash
wsl -d Ubuntu
cd /mnt/c/Users/<your-username>/Downloads/D-R-master/D-R-master
conda activate dr_env
jupyter notebook replications/replication_guide.ipynb
```

---

## Table of Contents
1. [Prerequisites](#1-prerequisites)
2. [Environment Setup](#2-environment-setup)
3. [API Keys](#3-api-keys)
4. [Data Preparation](#4-data-preparation)
5. [Multi-Agent Debate (10 questions)](#5-multi-agent-debate)
6. [Build Training Data](#6-build-training-data)
7. [SFT Training](#7-sft-training)
8. [T-DPO Training](#8-t-dpo-training)
9. [Merge & Convert to GGUF](#9-merge--convert-to-gguf)
10. [Evaluate Both Models](#10-evaluate-both-models)
11. [Results Summary](#11-results-summary)
12. [Robustness Test (5 questions)](#12-robustness-test-5-questions)

---
## 1. Prerequisites

Before starting, confirm you have:

| Requirement | Notes |
|---|---|
| Windows 10/11 with WSL2 | Install via `wsl --install` in admin PowerShell |
| Ubuntu distro | `wsl --install -d Ubuntu` |
| 16 GB RAM RECOMMENDED 
| API keys | OpenAI, Anthropic, Google Gemini |

In [ ]:
# Check system: free RAM and number of CPU threads
!free -h
!echo "CPU threads: $(nproc)"

---
## 2. Environment Setup

Run this **once**. It installs Miniconda (if needed) and creates the `dr_env` conda environment with all dependencies.

In [ ]:
# Run from the repo root
import os
os.chdir('/mnt/c/Users/claudio/Downloads/D-R-master/D-R-master')
!pwd

In [ ]:
# Install environment (skip if dr_env already exists)
!bash replications/setup_environment.sh

In [ ]:
# Verify key packages are installed
!conda run -n dr_env python -c "
import torch, transformers, peft, trl, datasets, openai, anthropic, llama_cpp
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('trl:', trl.__version__)
print('All packages OK')
"

---
## 3. Download Models

Two model files are required before anything can run:

| Model | Use | Size | How obtained |
|---|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` (HuggingFace) | SFT + T-DPO training | ~3 GB | Downloaded below (or auto on first train run) |
| `qwen2.5-1.5b-instruct-q4_k_m.gguf` | Debate student + base evaluation | ~1.1 GB | Downloaded below |

> ⏱ ~5–10 minutes depending on connection speed.


In [ ]:
# Download the base Qwen2.5-1.5B-Instruct GGUF (Q4) for llama.cpp inference
import os
os.makedirs('./models/Qwen2.5-1.5B-Instruct-GGUF', exist_ok=True)
gguf_path = './models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf'

if os.path.exists(gguf_path):
    print(f'Already exists: {gguf_path}')
else:
    print('Downloading GGUF...')
    !wget -q --show-progress \
        "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf" \
        -O {gguf_path}
    print('Done.')


In [ ]:
# Pre-download the HuggingFace weights for training (caches to ~/.cache/huggingface/)
print('Downloading Qwen2.5-1.5B-Instruct HF weights ...')

from transformers import AutoModelForCausalLM, AutoTokenizer

AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', torch_dtype='float32')
print('Download complete.')


---
## 3. API Keys

Set your API keys below. These are needed for the debate phase (GPT-4o, Claude, Gemini).

In [ ]:
import os

os.environ['OPENAI_API_KEY']    = ''    
os.environ['ANTHROPIC_API_KEY'] = ''  
os.environ['GEMINI_API_KEY']    = ''     

# Check keys are set.
for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY']:
    val = os.environ.get(key, '')
    status = 'SET' if val and not val.endswith('...') else 'NOT SET'
    print(f'{key}: {status}')

---
## 4. Data Preparation

Downloads MMLU-Pro Computer Science questions.

In [ ]:
!python get_and_split_data.py

---
## 5. Multi-Agent Debate

Runs a 4-agent debate across 4 rounds on 10 training questions.

| Agent | Model |
|---|---|
| Teacher 1 | GPT-4o (`gpt-4o-2024-08-06`) |
| Teacher 2 | Claude (`claude-opus-4-6`) |
| Teacher 3 | Gemini (`gemini-2.5-flash`) |
| Student | Qwen2.5-1.5B-Instruct Q4 (local, llama.cpp) |

The student model is served locally on port 58000 during the debate.

In [ ]:
# Start the student model server
import subprocess, time, requests

server = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', './models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf',
    '--host', '0.0.0.0', '--port', '58000',
    '--n_ctx', '4096', '--n_threads', str(os.cpu_count())
], stdout=open('/tmp/llama_debate.log', 'w'), stderr=subprocess.STDOUT)

print('Waiting for server..')
for _ in range(120):
    try:
        if requests.get('http://localhost:58000/v1/models', timeout=2).status_code == 200:
            print('Server ready.')
            break
    except:
        pass
    time.sleep(5)
else:
    print('ERROR: server did not start. Check /tmp/llama_debate.log')

In [ ]:
# Run the debate (10 questions)
!python debate_teachers_student_w_critique.py --dataset MMLUProComp

In [ ]:
# Stop the student model server
server.terminate()
server.wait()
print('Server stopped.')

In [ ]:
# Check debate output
!ls -lh data/MAG_new_mistral/MMLUProComp_1000.jsonl
!echo "Lines (= debate entries):"
!wc -l data/MAG_new_mistral/MMLUProComp_1000.jsonl

---
## 6. Build Training Data

Converts the debate output into two formats:
- **SFT**: chosen (correct) reasoning chains
- **T-DPO**: preference pairs (chosen vs rejected answers)

In [ ]:
# Build SFT data
!python mag2sft.py
!echo "SFT examples:"
!wc -l data/mag_new_sft_w_mistral/MMLUProComp_mag_new_974_sft_w_mistral.jsonl

In [ ]:
# Build T-DPO preference pairs
!python mag2preference.py
!echo "Preference pairs:"
!wc -l data/mag_new_preference_pair_w_mistral/MMLUProComp_mag_new_974_preference_pair_w_mistral.jsonl

---
## 7. SFT Training

Fine-tunes Qwen2.5-1.5B-Instruct using Supervised Fine-Tuning on the D&R chosen examples.

**Hyperparameters:**

| Parameter | Value |
|---|---|
| Base model | `Qwen/Qwen2.5-1.5B-Instruct` |
| Epochs | 2 |
| Learning rate | 2e-4 |
| Batch size | 1 (grad accum steps: 4) |
| Max sequence length | 512 tokens |
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| Optimizer | Adafactor |
| Dtype | float32 |
| Seed | 42 |

> ⏱ ~2–4 hours on CPU.

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES="" TOKENIZERS_PARALLELISM=false python sft.py \
    --dataset_name ./data/mag_new_sft_w_mistral/MMLUProComp_mag_new_974_sft_w_mistral.jsonl \
    --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
    --learning_rate 2.0e-4 \
    --num_train_epochs 2 \
    --packing false \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 4 \
    --max_seq_length 512 \
    --use_peft \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_task_type CAUSAL_LM \
    --logging_steps 0.1 \
    --eval_strategy no \
    --save_strategy steps \
    --save_steps 0.5 \
    --output_dir ./tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch \
    --optim adafactor \
    --torch_dtype float32 \
    --bf16 false \
    --fp16 false \
    --dataloader_num_workers 1 \
    --seed 42


In [ ]:
# Verify SFT adapter was saved
!ls tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch/

---
## 8. T-DPO Training

Applies Teacher-guided DPO on top of the SFT adapter using the preference pairs.

**Hyperparameters:**

| Parameter | Value |
|---|---|
| Epochs | 3 |
| Learning rate | 5e-6 |
| Batch size | 1 (grad accum steps: 8) |
| Max length | 512 tokens |
| Max prompt length | 256 tokens |
| Warmup ratio | 0.1 |
| Optimizer | Adafactor |
| Gradient checkpointing | Yes |
| Seed | 42 |

> ⏱ ~3–6 hours on CPU.

In [ ]:
%%bash
cd /mnt/c/Users/claudio/Downloads/D-R-master/D-R-master
CUDA_VISIBLE_DEVICES="" python tdpo_after_sft.py \
    --dataset_name ./data/mag_new_preference_pair_w_mistral/MMLUProComp_mag_new_974_preference_pair_w_mistral.jsonl \
    --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
    --learning_rate 5.0e-6 \
    --num_train_epochs 3 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --max_length 512 \
    --max_prompt_length 256 \
    --truncation_mode keep_end \
    --no_remove_unused_columns \
    --use_peft \
    --warmup_ratio 0.1 \
    --logging_steps 0.1 \
    --eval_strategy no \
    --save_strategy steps \
    --save_steps 0.1 \
    --output_dir ./tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch \
    --torch_dtype float32 \
    --dataloader_num_workers 0 \
    --precompute_ref_log_probs \
    --model_adapter_name tdpo \
    --ref_adapter_name reference \
    --sft_adapter_path ./tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch \
    --optim adafactor \
    --gradient_checkpointing \
    --seed 42


In [ ]:
# Verify T-DPO adapter was saved
!ls tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch/tdpo/

---
## 9. Merge & Convert to GGUF

Merges the LoRA adapter back into the base model weights, then converts to GGUF format for llama.cpp inference.

In [ ]:
# Merge adapter into base model
!python replications/merge_adapter.py \
    --adapter_path ./tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch \
    --base_model   Qwen/Qwen2.5-1.5B-Instruct \
    --output_dir   ./models/qwen-distilled-merged \
    --adapter_name tdpo

In [ ]:
# Convert to GGUF (f16)
LLAMA_CPP_DIR="${LLAMA_CPP_DIR:-./llama.cpp}"
!python {LLAMA_CPP_DIR}/convert_hf_to_gguf.py ./models/qwen-distilled-merged \
    --outfile ./models/qwen-distilled.gguf \
    --outtype f16

In [ ]:
# Verify GGUF was created
!ls -lh models/qwen-distilled.gguf

---
## 10. Evaluate Both Models

Evaluates both the base model and the distilled model on the 137-question test set.

**Evaluation settings:**

| Parameter | Value |
|---|---|
| Dataset | MMLU-Pro Computer Science |
| Test questions | 137 |
| Temperature | 0.3 |
| Top-p | 0.9 |
| Max new tokens | 700 |
| Server port | 58000 |

In [ ]:
# Evaluate the BASE model
import os, subprocess, time, requests

base_server = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', './models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf',
    '--host', '0.0.0.0', '--port', '58000',
    '--n_ctx', '4096', '--n_threads', str(os.cpu_count())
], stdout=open('/tmp/eval_base.log', 'w'), stderr=subprocess.STDOUT)

print('Waiting for base model server...')
for _ in range(120):
    try:
        if requests.get('http://localhost:58000/v1/models', timeout=2).status_code == 200:
            print('Server ready.')
            break
    except:
        pass
    time.sleep(5)


In [ ]:
!python test.py \
    --dataset MMLUProComp \
    --model_name qwen2.5-1.5b-instruct-q4_k_m \
    --num_proc 1 \
    --max_new_tokens 700 \
    --temperature 0.3 \
    --top_p 0.9 \
    --port 58000

In [ ]:
base_server.terminate(); base_server.wait(); print('Base model server stopped.')

In [ ]:
# Evaluate the DISTILLED model
import os, subprocess, time, requests

dist_server = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', './models/qwen-distilled.gguf',
    '--host', '0.0.0.0', '--port', '58000',
    '--n_ctx', '4096', '--n_threads', str(os.cpu_count())
], stdout=open('/tmp/eval_distilled.log', 'w'), stderr=subprocess.STDOUT)

print('Waiting for distilled model server...')
for _ in range(120):
    try:
        if requests.get('http://localhost:58000/v1/models', timeout=2).status_code == 200:
            print('Server ready.')
            break
    except:
        pass
    time.sleep(5)


In [ ]:
!python test.py \
    --dataset MMLUProComp \
    --model_name distilled-qwen \
    --num_proc 1 \
    --max_new_tokens 700 \
    --temperature 0.3 \
    --top_p 0.9 \
    --port 58000

In [ ]:
dist_server.terminate(); dist_server.wait(); print('Distilled model server stopped.')

---
## 11. Results Summary

Results obtained in this experiment (2026-04-26, seed=42, 10-question debate):

| Run | Model | Debate Qs | Correct | Total | Accuracy |
|---|---|---|---|---|---|
| Base | Qwen2.5-1.5B-Instruct Q4 | — | 24 | 137 | **17.52%** |
| 10q distilled | Qwen2.5-1.5B-Instruct (SFT+T-DPO) | 10 | 31 | 137 | **22.63%** |
| 5q distilled | Qwen2.5-1.5B-Instruct (SFT+T-DPO) | 5 | — | 137 | *(see experiment log)* |

The distilled model (10q) improves over the base by **+5.11 percentage points**.

Results may vary slightly between runs due to temperature=0.3 sampling over 137 questions.

---
## 12. Robustness Test (5 questions)

Repeats the pipeline with only 5 debate questions to test whether the improvement holds with less training data.

All outputs use a `_5q` suffix — the 10q results are not overwritten.

> Run this only after completing Steps 4–10 above and backing up the 10q artifacts.

In [ ]:
%%bash
# Back up 10q artifacts before running the 5q stress test
cd /mnt/c/Users/claudio/Downloads/D-R-master/D-R-master
mv tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch \
   tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch_10q 2>/dev/null || echo 'already renamed'
mv tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch \
   tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch_10q 2>/dev/null || echo 'already renamed'
mv models/qwen-distilled-merged models/qwen-distilled-merged-10q 2>/dev/null || echo 'already renamed'
mv models/qwen-distilled.gguf   models/qwen-distilled-10q.gguf   2>/dev/null || echo 'already renamed'
cp -r tmp_test tmp_test_10q 2>/dev/null || echo 'already copied'
echo 'Backup done.'


In [ ]:
# Run the full 5q pipeline (debate → SFT → T-DPO → merge → GGUF)
!bash replications/run_pipeline_5q.sh

In [ ]:
# Evaluate the 5q distilled model
!bash replications/evaluate_5q.sh